### Problems with the `NeuralNetwork_v3` model we've built in [nn_module.ipynb](nn_module.ipynb) 

- No transformations can be applied on training data
- No shuffling and  sampling support when using Mini-Batch Gradient Descent
- No batch management
- No support for parallel data loading/processing

In [550]:
from sklearn.datasets import make_classification
import torch
from torch.utils.data import DataLoader, Dataset

In [551]:
X,y=make_classification(
    n_samples=10,
    n_features=5,
    n_classes=2,
    random_state=2026
)

In [552]:
## data shape
X.shape,y.shape

((10, 5), (10,))

In [553]:
X[:5]

array([[-0.30864091,  0.11208642,  0.14635386, -0.48931261,  0.46107161],
       [ 0.43638303, -0.83506663, -0.4333855 ,  1.01814495, -1.25096854],
       [-0.88898552, -0.76412765, -0.06904386, -0.8851414 ,  0.36560731],
       [ 0.35605381, -1.05474615,  0.1365157 ,  1.01081177, -1.35130518],
       [ 1.36984779,  1.08663717,  0.94003573,  1.40772394, -0.64377853]])

In [554]:
y[:5]

array([0, 1, 0, 1, 1])

In [555]:
## type casting
X=torch.tensor(X)
y=torch.tensor(y)

In [556]:
class custom_dataset(Dataset):
    def __init__(self,features,labels):
        self.feature=features
        self.label=labels
    def __len__(self):
        return len(self.feature) 
    def __getitem__(self, index):
        return self.feature[index], self.label[index]

In [557]:
ds=custom_dataset(X,y)

In [558]:
len(ds)

10

In [559]:
ds[2]

(tensor([-0.8890, -0.7641, -0.0690, -0.8851,  0.3656], dtype=torch.float64),
 tensor(0, dtype=torch.int32))

In [560]:
dataloader=DataLoader(ds,batch_size=5,shuffle=True)

In [561]:
data=iter(dataloader) ## dont need to do, but do for safety

In [562]:
for batch_x, batch_y in data:
    print(batch_x,batch_y)

tensor([[ 1.3698,  1.0866,  0.9400,  1.4077, -0.6438],
        [-0.4900,  0.9124, -1.1665, -1.1310,  1.3822],
        [ 0.3561, -1.0547,  0.1365,  1.0108, -1.3513],
        [ 0.2036,  2.9703,  1.4768, -1.1455,  2.3913],
        [ 0.2606, -1.3449, -0.0101,  1.0161, -1.4963]], dtype=torch.float64) tensor([1, 0, 1, 0, 1], dtype=torch.int32)
tensor([[-1.2293, -1.3841, -0.0059, -1.0660,  0.2156],
        [ 0.4364, -0.8351, -0.4334,  1.0181, -1.2510],
        [-0.0945, -0.6941,  1.6728,  0.2015, -0.5038],
        [-0.8890, -0.7641, -0.0690, -0.8851,  0.3656],
        [-0.3086,  0.1121,  0.1464, -0.4893,  0.4611]], dtype=torch.float64) tensor([0, 1, 1, 0, 0], dtype=torch.int32)


#### Performing transformation in data

In [563]:
class custom_dataset(Dataset):
    def __init__(self,features,labels):
        self.feature=features
        self.label=labels
    def __len__(self):
        return len(self.feature) 
    def __getitem__(self, index): ## do inside this method
        self.feature[index]*=2
        return self.feature[index], self.label[index]

#### Performing parallelization

In [564]:
dataloader=DataLoader(ds,batch_size=5,shuffle=True,num_workers=3) ## num_workers enable 3 core to work in dataset

In [565]:
""" drop_last: say we have 32 rows of data and we have batch size of 10 so batch would be 10 10 10 2 
drop last enables us to use than last uncomplete batch"2" or not
"""

' drop_last: say we have 32 rows of data and we have batch size of 10 so batch would be 10 10 10 2 \ndrop last enables us to use than last uncomplete batch"2" or not\n'

#### improving our model

In [566]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [567]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [568]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [569]:
## label encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [570]:
## torch typecasting
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [571]:
class custom_dataset(Dataset):
    def __init__(self,features,labels):
        self.feature=features
        self.label=labels
    def __len__(self):
        return len(self.feature) 
    def __getitem__(self, index):
        return self.feature[index], self.label[index]

In [572]:
train_ds=custom_dataset(X_train_tensor,y_train_tensor)
test_ds=custom_dataset(X_test_tensor,y_test_tensor)

In [573]:
train_ds_loader=DataLoader(train_ds,batch_size=20,shuffle=True)
test_ds_loader=DataLoader(test_ds,batch_size=20,shuffle=True)

In [574]:
import torch.nn as nn

In [575]:
class NeuralNetwork_v4(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.container=nn.Sequential(
            nn.Linear(features,3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid())
    def forward(self,inp):
        return self.container(inp)    

In [576]:
## defininf param
LR=0.1
epocs=50

In [ ]:
model=NeuralNetwork_v4(X_train_tensor.shape[1])
optimizer=torch.optim.SGD(model.parameters(),lr=LR)
loss=nn.BCELoss()

In [ ]:
for i in range(epocs):
    for x_batch, y_batch in train_ds_loader:
        ## forward pass 
        y_pred=model.forward(x_batch)
        ## calculate loss
        loss_cal=loss(y_pred,y_batch.reshape(-1,1))

        ## backpropagation
        optimizer.zero_grad()
        loss_cal.backward()
        optimizer.step()

In [578]:
model.eval()
accuracy=[]

with torch.no_grad():
    for x_batch, y_batch in test_ds_loader:
        y_pred=model(x_batch)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions
         # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == y_batch).float().mean().item()
        accuracy.append(batch_accuracy)

avg_acc=sum(accuracy)/len(accuracy)
print(f"Accuracy is {avg_acc:.2f}")

Accuracy is 0.97
